In [1]:
dev = "xxx"  # key hidden

1. Function to scrape youtube comments from a given Video ID

In [2]:
import googleapiclient.discovery
import pandas as pd

api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY = dev

youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)


def getcomments(video):
  request = youtube.commentThreads().list(
      part="snippet",
      videoId=video,
      maxResults=100
  )

  comments = []

  # Execute the request.
  response = request.execute()

  # Get the comments from the response.
  for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      public = item['snippet']['isPublic']
      comments.append([
          comment['authorDisplayName'],
          comment['publishedAt'],
          comment['likeCount'],
          comment['textOriginal'],
          comment['videoId'],
          public
      ])

  while (1 == 1):
    try:
     nextPageToken = response['nextPageToken']
    except KeyError:
     break
    nextPageToken = response['nextPageToken']
    # Create a new request object with the next page token.
    nextRequest = youtube.commentThreads().list(part="snippet", videoId=video, maxResults=100, pageToken=nextPageToken)
    # Execute the next request.
    response = nextRequest.execute()
    # Get the comments from the next response.
    for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      public = item['snippet']['isPublic']
      comments.append([
          comment['authorDisplayName'],
          comment['publishedAt'],
          comment['likeCount'],
          comment['textOriginal'],
          comment['videoId'],
          public
      ])

  df2 = pd.DataFrame(comments, columns=['author', 'updated_at', 'like_count', 'text','video_id','public'])
  return df2

2. Scrape the Video ID's of top n Youtube videos about a given Topic (Most Relevant)

In [3]:
youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)

def get_top_video_ids(topic, max_results=5):
    request = youtube.search().list(
        part="snippet",
        q=topic,
        type="video",
        maxResults=max_results,
        order="relevance"
    )

    response = request.execute()

    video_ids = []

    for item in response["items"]:
        video_ids.append(item["id"]["videoId"])

    return video_ids

In [4]:
topics = [
    "interstellar ending explained",
    "interstellar theory",
    "interstellar explained",
    "interstellar movie analysis",
    "interstellar ending breakdown"
]

all_video_ids = []

for topic in topics:

    print("Searching:", topic)

    video_ids = get_top_video_ids(topic, 15)

    all_video_ids.extend(video_ids)


# remove duplicate video IDs
all_video_ids = list(set(all_video_ids))

print("Total unique videos:", len(all_video_ids))

all_video_ids

Searching: interstellar ending explained


HttpError: <HttpError 400 when requesting https://youtube.googleapis.com/youtube/v3/search?part=snippet&q=interstellar+ending+explained&type=video&maxResults=15&order=relevance&key=xxx&alt=json returned "API key not valid. Please pass a valid API key.". Details: "[{'message': 'API key not valid. Please pass a valid API key.', 'domain': 'global', 'reason': 'badRequest'}]">

In [8]:
all_dfs = []

for video in all_video_ids:
    df = getcomments(video)
    all_dfs.append(df)

comments_df = pd.concat(all_dfs, ignore_index=True)
comments_df.head()

ValueError: No objects to concatenate

In [9]:
comments_df.info()

NameError: name 'comments_df' is not defined

Changing the order of the comments according to the Likes

In [10]:
comments_df = comments_df.sort_values(
    by="like_count",
    ascending=False
)

NameError: name 'comments_df' is not defined

In [11]:
pd.set_option('display.max_colwidth', None)

comments_df.sort_values(
    by="like_count",
    ascending=False
)[["like_count", "text"]].head(20)

NameError: name 'comments_df' is not defined

Convert the Dataframe to CSV file as raw data

In [12]:
import pandas as pd
import re
import html

comments_df = pd.read_csv("raw_data.csv")

Inspect dataset

In [13]:
comments_df.head()

,author,updated_at,like_count,text,video_id,public
0,@kingsman1713,2020-03-30T00:09:44Z,41736,The guy who stay in the ship for 20 years.\n\nHe's the king of quarantine.,j3DuONZb3Ik,True
1,@IkanGelamaKuning,2021-08-18T15:33:15Z,37032,"A husband waiting for his wife shopping feels a very long time, while she feel only few minutes. A real time dilation.",JqKa6qyVYgg,True
2,@stevedevries2891,2021-01-05T01:33:47Z,17975,My wife keeps asking me to dust my office. I'm like - what if my dad wants to talk to me?,j3DuONZb3Ik,True
3,@joshfromdundermifflin8973,2020-03-05T18:54:19Z,15427,"The scene where cooper comes back to the ship after the ocean planet incident , and sees all the video transmissions from his children gets me every time.",j3DuONZb3Ik,True
4,@Dcook85,2014-11-24T01:48:48Z,15359,I was disappointed to see that Christopher Nolan did not add a disclaimer to the movie warning people not to go into black holes. Now everyone's going to be doing it.,R1cexcjdyIE,True


In [14]:
comments_df.isnull().sum()

,0
author,0
updated_at,0
like_count,0
text,7
video_id,0
public,0


In [15]:
comments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54427 entries, 0 to 54426
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   author      54427 non-null  object
 1   updated_at  54427 non-null  object
 2   like_count  54427 non-null  int64 
 3   text        54420 non-null  object
 4   video_id    54427 non-null  object
 5   public      54427 non-null  bool  
dtypes: bool(1), int64(1), object(4)
memory usage: 2.1+ MB


Standardizing column names

In [16]:
comments_df.columns = comments_df.columns.str.lower().str.strip()
comments_df.columns

Index(['author', 'updated_at', 'like_count', 'text', 'video_id', 'public'], dtype='object')

Removing missing comments

In [17]:
comments_df = comments_df.dropna(subset=["text"])

Removing exact duplicates

In [18]:
comments_df = comments_df.drop_duplicates()

Remove duplicate comments

In [19]:
comments_df = comments_df.drop_duplicates(subset=["text"])

Decoding HTML entities

In [20]:
comments_df["text"] = comments_df["text"].apply(html.unescape)

Removing URLs

In [21]:
comments_df["text"] = comments_df["text"].apply(lambda x: re.sub(r"http\S+|www\S+", "", x))

Removing extra white spaces

In [22]:
comments_df["text"] = comments_df["text"].apply(lambda x: re.sub(r"\s+", " ", x))

Strip spaces

In [23]:
comments_df["text"] = comments_df["text"].str.strip()

Create char_count column

In [24]:
comments_df["char_count"] = comments_df["text"].apply(len)

Create word_count column

In [25]:
comments_df["word_count"] = comments_df["text"].apply(lambda x: len(x.split()))

Create sentence_count column

In [26]:
comments_df["sentence_count"] = comments_df["text"].apply(lambda x: len(re.findall(r"[.!?]+", x)))

Detect spam comments

In [27]:
spam_words = [
    "subscribe",
    "check my channel",
    "follow me",
    "click here",
    "free",
    "giveaway"
]

comments_df["spam_flag"] = "no"
def detect_spam(comment):

    comment = str(comment).lower()

    for word in spam_words:
        if word in comment:
            return "yes"

    return "no"

comments_df["spam_flag"] = comments_df["text"].apply(detect_spam)
comments_df = comments_df[comments_df["spam_flag"] == "no"]

Remove extremely short comments

In [28]:
comments_df = comments_df[comments_df["word_count"] >= 2]

Final validation checks

In [30]:
print(comments_df.head())


                       author            updated_at  like_count  \
0               @kingsman1713  2020-03-30T00:09:44Z       41736   
1           @IkanGelamaKuning  2021-08-18T15:33:15Z       37032   
2           @stevedevries2891  2021-01-05T01:33:47Z       17975   
3  @joshfromdundermifflin8973  2020-03-05T18:54:19Z       15427   
4                    @Dcook85  2014-11-24T01:48:48Z       15359   

                                                                                                                                                                     text  \
0                                                                                                 The guy who stay in the ship for 20 years. He's the king of quarantine.   
1                                                  A husband waiting for his wife shopping feels a very long time, while she feel only few minutes. A real time dilation.   
2                                                                             

In [31]:
print(comments_df.info())


<class 'pandas.core.frame.DataFrame'>
Index: 51345 entries, 0 to 54426
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   author          51345 non-null  object
 1   updated_at      51345 non-null  object
 2   like_count      51345 non-null  int64 
 3   text            51345 non-null  object
 4   video_id        51345 non-null  object
 5   public          51345 non-null  bool  
 6   char_count      51345 non-null  int64 
 7   word_count      51345 non-null  int64 
 8   sentence_count  51345 non-null  int64 
 9   spam_flag       51345 non-null  object
dtypes: bool(1), int64(4), object(5)
memory usage: 4.0+ MB
None


In [32]:
print(comments_df.isnull().sum())


author            0
updated_at        0
like_count        0
text              0
video_id          0
public            0
char_count        0
word_count        0
sentence_count    0
spam_flag         0
dtype: int64


In [33]:
print(comments_df.shape)

(51345, 10)


Export cleaned comments.csv

In [34]:
comments_df.to_csv("cleaned_comments.csv", index=False)